In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Global variables
FILE_NM_PREFIX = 'cleaned_data_'

In [2]:
dfs = {}

for i in range(2021, 2025):
    dfs[i] = pd.read_csv(FILE_NM_PREFIX + str(i) + '.csv')

In [3]:
dfs[2021].shape

(662, 7)

In [4]:
merged_df = pd.DataFrame()
merged_df[['OCC_CODE', 'OCC_TITLE', 'AIOE', 'TOT_EMP_21', 'A_PCT25_21', 'A_MEDIAN_21', 'A_PCT75_21']] = dfs[2021][['OCC_CODE', 'OCC_TITLE', 'AIOE', 'TOT_EMP', 'A_PCT25', 'A_MEDIAN', 'A_PCT75']]

for i in range(2022, 2025):
    yr = i - 2000
    df = pd.DataFrame()
    df[['OCC_CODE', f'TOT_EMP_{yr}', f'A_PCT25_{yr}', f'A_MEDIAN_{yr}', f'A_PCT75_{yr}']] = dfs[i][['OCC_CODE', 'TOT_EMP', 'A_PCT25', 'A_MEDIAN', 'A_PCT75']]
    merged_df = merged_df.merge(df, how='inner', on='OCC_CODE')

merged_df.head(5)

,OCC_CODE,OCC_TITLE,AIOE,TOT_EMP_21,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,TOT_EMP_22,A_PCT25_22,A_MEDIAN_22,A_PCT75_22,TOT_EMP_23,A_PCT25_23,A_MEDIAN_23,A_PCT75_23,TOT_EMP_24,A_PCT25_24,A_MEDIAN_24,A_PCT75_24
0,11-1021,General and Operations Managers,0.574877,2984920,60690.0,97970.0,151750.0,3376680,62070.0,98100.0,154560.0,3507810,65180.0,101280.0,160290.0,3584420,67160.0,102950.0,164130.0
1,11-2011,Advertising and Promotions Managers,1.294387,22520,94020.0,127150.0,172210.0,22010,85020.0,127830.0,172920.0,20630,88810.0,131870.0,188530.0,21100,85990.0,126960.0,178570.0
2,11-2021,Marketing Managers,1.315032,278690,100010.0,135030.0,192520.0,328570,103060.0,140040.0,198530.0,368940,108000.0,157620.0,208000.0,384980,111210.0,161030.0,211080.0
3,11-2022,Sales Managers,1.266280,453800,84450.0,127490.0,173010.0,536390,87910.0,130600.0,180390.0,575880,93540.0,135160.0,196700.0,603710,95910.0,138060.0,201490.0
4,11-3021,Computer and Information Systems Managers,1.059853,485190,122820.0,159010.0,198750.0,533220,127180.0,164070.0,207850.0,592600,131770.0,169510.0,214050.0,645970,134350.0,171200.0,216220.0


In [5]:
aioe_col = list(merged_df['AIOE'])

print(len(aioe_col))

aioe_25_percentile = np.percentile(aioe_col, 25)
aioe_75_percentile = np.percentile(aioe_col, 75)

aioe_33_percentile = np.percentile(aioe_col, 33.3)
aioe_67_percentile = np.percentile(aioe_col, 66.7)

print(aioe_25_percentile, aioe_75_percentile)
print(aioe_33_percentile, aioe_67_percentile)

661
-0.8863602 1.02251
-0.6913620240000001 0.7077313600000003


In [6]:
merged_df['AIOE_CAT'] = 'M'

merged_df.loc[merged_df['AIOE'] <= aioe_25_percentile, 'AIOE_CAT'] = 'L'
merged_df.loc[merged_df['AIOE'] > aioe_75_percentile, 'AIOE_CAT'] = 'H'

# merged_df.loc[merged_df['AIOE'] <= aioe_33_percentile, 'AIOE_CAT'] = 'L'
# merged_df.loc[merged_df['AIOE'] > aioe_67_percentile, 'AIOE_CAT'] = 'H'

merged_df.head()

,OCC_CODE,OCC_TITLE,AIOE,TOT_EMP_21,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,TOT_EMP_22,A_PCT25_22,A_MEDIAN_22,A_PCT75_22,TOT_EMP_23,A_PCT25_23,A_MEDIAN_23,A_PCT75_23,TOT_EMP_24,A_PCT25_24,A_MEDIAN_24,A_PCT75_24,AIOE_CAT
0,11-1021,General and Operations Managers,0.574877,2984920,60690.0,97970.0,151750.0,3376680,62070.0,98100.0,154560.0,3507810,65180.0,101280.0,160290.0,3584420,67160.0,102950.0,164130.0,M
1,11-2011,Advertising and Promotions Managers,1.294387,22520,94020.0,127150.0,172210.0,22010,85020.0,127830.0,172920.0,20630,88810.0,131870.0,188530.0,21100,85990.0,126960.0,178570.0,H
2,11-2021,Marketing Managers,1.315032,278690,100010.0,135030.0,192520.0,328570,103060.0,140040.0,198530.0,368940,108000.0,157620.0,208000.0,384980,111210.0,161030.0,211080.0,H
3,11-2022,Sales Managers,1.266280,453800,84450.0,127490.0,173010.0,536390,87910.0,130600.0,180390.0,575880,93540.0,135160.0,196700.0,603710,95910.0,138060.0,201490.0,H
4,11-3021,Computer and Information Systems Managers,1.059853,485190,122820.0,159010.0,198750.0,533220,127180.0,164070.0,207850.0,592600,131770.0,169510.0,214050.0,645970,134350.0,171200.0,216220.0,H


In [7]:
cols = ['TOT_EMP', 'A_PCT25', 'A_MEDIAN', 'A_PCT75']

for col in cols:
    for i in range(21, 25):
        if col != 'TOT_EMP':
            merged_df[f"Product_{col}_{i}"] = merged_df[f"{col}_{i}"] * merged_df[f"TOT_EMP_{i}"]
        if i != 21:
            merged_df[f"{col}_YoY_{i}"] = merged_df[f"{col}_{i}"] / merged_df[f"{col}_{i - 1}"] - 1
        
    merged_df[f"{col}_22-24"] = merged_df[f"{col}_24"] / merged_df[f"{col}_22"] - 1
    merged_df[f"{col}_21-24"] = merged_df[f"{col}_24"] / merged_df[f"{col}_21"] - 1

for i in range(21, 25):
    merged_df[f"dispersion_ratio_{i}"] = (merged_df[f"A_PCT75_{i}"] - merged_df[f"A_PCT25_{i}"]) / merged_df[f"A_MEDIAN_{i}"]

merged_df['dispersion_ratio_change_22-24'] = merged_df['dispersion_ratio_24'] / merged_df['dispersion_ratio_22'] - 1
        
merged_df.head()

,OCC_CODE,OCC_TITLE,AIOE,TOT_EMP_21,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,TOT_EMP_22,A_PCT25_22,A_MEDIAN_22,...,A_PCT75_YoY_23,Product_A_PCT75_24,A_PCT75_YoY_24,A_PCT75_22-24,A_PCT75_21-24,dispersion_ratio_21,dispersion_ratio_22,dispersion_ratio_23,dispersion_ratio_24,dispersion_ratio_change_22-24
0,11-1021,General and Operations Managers,0.574877,2984920,60690.0,97970.0,151750.0,3376680,62070.0,98100.0,...,0.037073,5.883109e+11,0.023957,0.061918,0.081582,0.929468,0.942813,0.939080,0.941914,-0.000954
1,11-2011,Advertising and Promotions Managers,1.294387,22520,94020.0,127150.0,172210.0,22010,85020.0,127830.0,...,0.090273,3.767827e+09,-0.052830,0.032674,0.036932,0.614943,0.687632,0.756199,0.729206,0.060460
2,11-2021,Marketing Managers,1.315032,278690,100010.0,135030.0,192520.0,328570,103060.0,140040.0,...,0.047701,8.126158e+10,0.014808,0.063215,0.096406,0.685107,0.681734,0.634437,0.620195,-0.090268
3,11-2022,Sales Managers,1.266280,453800,84450.0,127490.0,173010.0,536390,87910.0,130600.0,...,0.090415,1.216415e+11,0.024352,0.116969,0.164615,0.694643,0.708116,0.763244,0.764740,0.079964
4,11-3021,Computer and Information Systems Managers,1.059853,485190,122820.0,159010.0,198750.0,533220,127180.0,164070.0,...,0.029829,1.396716e+11,0.010138,0.040269,0.087899,0.477517,0.491680,0.485399,0.478213,-0.027391


In [8]:
merged_df.assign(TOT_EMP_CHANGE=merged_df['TOT_EMP_22'] - merged_df['TOT_EMP_21']) \
         .sort_values(by='TOT_EMP_CHANGE', ascending=False) \
         .head(5)

,OCC_CODE,OCC_TITLE,AIOE,TOT_EMP_21,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,TOT_EMP_22,A_PCT25_22,A_MEDIAN_22,...,Product_A_PCT75_24,A_PCT75_YoY_24,A_PCT75_22-24,A_PCT75_21-24,dispersion_ratio_21,dispersion_ratio_22,dispersion_ratio_23,dispersion_ratio_24,dispersion_ratio_change_22-24,TOT_EMP_CHANGE
0,11-1021,General and Operations Managers,0.574877,2984920,60690.0,97970.0,151750.0,3376680,62070.0,98100.0,...,5.883109e+11,0.023957,0.061918,0.081582,0.929468,0.942813,0.939080,0.941914,-0.000954,391760
313,35-3031,Waiters and Waitresses,-0.948100,1804030,20020.0,26000.0,30850.0,2122210,21810.0,29120.0,...,1.044270e+11,0.090144,0.245879,0.470016,0.416538,0.501030,0.558234,0.582346,0.162297,318180
653,53-7062,"Laborers and Freight, Stock, and Material Move...",-1.709183,2729010,29240.0,31230.0,37970.0,2934050,31130.0,36110.0,...,1.382999e+11,0.031591,0.091830,0.221227,0.279539,0.314040,0.289697,0.281459,-0.103750,205040
305,35-1012,First-Line Supervisors of Food Preparation and...,-0.257549,1040600,29430.0,36570.0,46840.0,1169620,31200.0,37050.0,...,6.046546e+10,0.046015,0.090131,0.087105,0.476073,0.418623,0.368640,0.369436,-0.117499,129020
312,35-3011,Bartenders,-0.615988,485330,20560.0,26350.0,34120.0,613070,22340.0,29380.0,...,3.488709e+10,0.093224,0.245740,0.371336,0.514611,0.518039,0.601079,0.626305,0.208990,127740


In [9]:
agg_df = merged_df.groupby('AIOE_CAT')[['TOT_EMP_21', 'Product_A_PCT25_21', 'Product_A_MEDIAN_21', 'Product_A_PCT75_21',
                                        'TOT_EMP_22', 'Product_A_PCT25_22', 'Product_A_MEDIAN_22', 'Product_A_PCT75_22',
                                        'TOT_EMP_23', 'Product_A_PCT25_23', 'Product_A_MEDIAN_23', 'Product_A_PCT75_23',
                                        'TOT_EMP_24', 'Product_A_PCT25_24', 'Product_A_MEDIAN_24', 'Product_A_PCT75_24']].sum()

agg_df.loc['Total'] = agg_df.sum(axis=0)

for i in range(21, 25):
    for col in cols:
        if col != 'TOT_EMP':
            agg_df[f"{col}_{i}"] = agg_df[f"Product_{col}_{i}"] / agg_df[f"TOT_EMP_{i}"]
            agg_df.drop(f"Product_{col}_{i}", axis=1, inplace=True)

agg_df

,TOT_EMP_21,TOT_EMP_22,TOT_EMP_23,TOT_EMP_24,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,A_PCT25_22,A_MEDIAN_22,A_PCT75_22,A_PCT25_23,A_MEDIAN_23,A_PCT75_23,A_PCT25_24,A_MEDIAN_24,A_PCT75_24
AIOE_CAT,,,,,,,,,,,,,,,,
H,27255570.0,28493630.0,29278350.0,29742960.0,54678.307913,72144.800600,96154.360947,57618.364213,75337.582081,100530.333706,61031.952108,80129.083114,106127.247171,63852.130746,83259.458268,109832.504246
L,27899730.0,29512580.0,29972850.0,30340040.0,31386.043356,38242.711044,46711.932098,33474.853259,40220.172567,49248.924242,35735.034947,42769.427278,51952.641414,37312.229977,44592.956895,54074.996246
M,57636230.0,59905060.0,61290560.0,62072910.0,39948.780730,51031.872694,65748.763854,42315.682672,53463.066611,68730.532833,45300.187409,56463.665113,72224.584582,47169.729618,58928.282141,74884.418912
Total,112791530.0,117911270.0,120541760.0,122155910.0,41390.055606,52970.233912,68387.254651,43800.810095,55434.484443,71538.909234,46742.884388,58806.662715,75418.549142,48783.299163,61292.038892,78225.237937


In [10]:
for i in range(21, 25):
    agg_df[f"dispersion_ratio_{i}"] = (agg_df[f"A_PCT75_{i}"] - agg_df[f"A_PCT25_{i}"]) / agg_df[f"A_MEDIAN_{i}"]

agg_df[f"dispersion_change_22-24"] = (agg_df["dispersion_ratio_24"] - agg_df["dispersion_ratio_22"]) / agg_df["dispersion_ratio_22"]

agg_df

,TOT_EMP_21,TOT_EMP_22,TOT_EMP_23,TOT_EMP_24,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,A_PCT25_22,A_MEDIAN_22,A_PCT75_22,...,A_MEDIAN_23,A_PCT75_23,A_PCT25_24,A_MEDIAN_24,A_PCT75_24,dispersion_ratio_21,dispersion_ratio_22,dispersion_ratio_23,dispersion_ratio_24,dispersion_change_22-24
AIOE_CAT,,,,,,,,,,,,,,,,,,,,,
H,27255570.0,28493630.0,29278350.0,29742960.0,54678.307913,72144.800600,96154.360947,57618.364213,75337.582081,100530.333706,...,80129.083114,106127.247171,63852.130746,83259.458268,109832.504246,0.574900,0.569596,0.562783,0.552254,-0.030446
L,27899730.0,29512580.0,29972850.0,30340040.0,31386.043356,38242.711044,46711.932098,33474.853259,40220.172567,49248.924242,...,42769.427278,51952.641414,37312.229977,44592.956895,54074.996246,0.400753,0.392193,0.379187,0.375906,-0.041528
M,57636230.0,59905060.0,61290560.0,62072910.0,39948.780730,51031.872694,65748.763854,42315.682672,53463.066611,68730.532833,...,56463.665113,72224.584582,47169.729618,58928.282141,74884.418912,0.505566,0.494077,0.476845,0.470312,-0.048099
Total,112791530.0,117911270.0,120541760.0,122155910.0,41390.055606,52970.233912,68387.254651,43800.810095,55434.484443,71538.909234,...,58806.662715,75418.549142,48783.299163,61292.038892,78225.237937,0.509667,0.500376,0.487626,0.480355,-0.040012


In [11]:
agg_growth = pd.DataFrame()

cols = ['TOT_EMP', 'A_PCT25', 'A_MEDIAN', 'A_PCT75']

for col in cols:
    agg_growth[f"{col}_GROWTH_21-24"] = agg_df[f"{col}_24"] / agg_df[f"{col}_21"] - 1
    agg_growth[f"{col}_GROWTH_22-24"] = agg_df[f"{col}_24"] / agg_df[f"{col}_22"] - 1

    for i in range(22, 25):
        agg_growth[f"{col}_YOY_{i}"] = agg_df[f"{col}_{i}"] / agg_df[f"{col}_{i - 1}"] - 1

agg_growth

,TOT_EMP_GROWTH_21-24,TOT_EMP_GROWTH_22-24,TOT_EMP_YOY_22,TOT_EMP_YOY_23,TOT_EMP_YOY_24,A_PCT25_GROWTH_21-24,A_PCT25_GROWTH_22-24,A_PCT25_YOY_22,A_PCT25_YOY_23,A_PCT25_YOY_24,A_MEDIAN_GROWTH_21-24,A_MEDIAN_GROWTH_22-24,A_MEDIAN_YOY_22,A_MEDIAN_YOY_23,A_MEDIAN_YOY_24,A_PCT75_GROWTH_21-24,A_PCT75_GROWTH_22-24,A_PCT75_YOY_22,A_PCT75_YOY_23,A_PCT75_YOY_24
AIOE_CAT,,,,,,,,,,,,,,,,,,,,
H,0.091262,0.043846,0.045424,0.027540,0.015869,0.167778,0.108191,0.053770,0.059245,0.046208,0.154060,0.105152,0.044255,0.063600,0.039067,0.142252,0.092531,0.045510,0.055674,0.034913
L,0.087467,0.028038,0.057809,0.015596,0.012251,0.188816,0.114635,0.066552,0.067519,0.044136,0.166051,0.108721,0.051708,0.063382,0.042636,0.157627,0.097993,0.054311,0.054899,0.040852
M,0.076977,0.036188,0.039365,0.023128,0.012765,0.180755,0.114710,0.059248,0.070530,0.041270,0.154735,0.102224,0.047641,0.056125,0.043650,0.138948,0.089536,0.045351,0.050837,0.036827
Total,0.083024,0.035999,0.045391,0.022309,0.013391,0.178624,0.113753,0.058245,0.067169,0.043652,0.157103,0.105666,0.046521,0.060832,0.042264,0.143857,0.093464,0.046085,0.054231,0.037215


In [12]:
merged_h = merged_df[merged_df['AIOE_CAT'] == 'H']
merged_m = merged_df[merged_df['AIOE_CAT'] == 'M']
merged_l = merged_df[merged_df['AIOE_CAT'] == 'L']

labels = ['H', 'M', 'L']
mergeds = [merged_h, merged_m, merged_l]
attr = 'TOT_EMP_YoY'

for i in range(22, 25):
    for df, l in zip(mergeds, labels):
        df = df[f"{attr}_{i}"]
        print(f"In Year {i}, AIOE-{l}'s mean is {df.mean():.4f}, median is {df.median():.4f}, standard deviation is {df.std():.4f}")

    t_stat, p_welch = stats.ttest_ind(merged_h[f"{attr}_{i}"], merged_m[f"{attr}_{i}"], equal_var=False)
    print(t_stat, p_welch)
    t_stat, p_welch = stats.ttest_ind(merged_l[f"{attr}_{i}"], merged_m[f"{attr}_{i}"], equal_var=False)
    print(t_stat, p_welch)
    t_stat, p_welch = stats.ttest_ind(merged_h[f"{attr}_{i}"], merged_l[f"{attr}_{i}"], equal_var=False)
    print(t_stat, p_welch)
    print()

for df, l in zip(mergeds, labels):
    df = df['TOT_EMP_22-24']
    print(f"In Year 22-24, AIOE-{l}'s mean is {df.mean():.4f}, median is {df.median():.4f}, standard deviation is {df.std():.4f}")
t_stat, p_welch = stats.ttest_ind(merged_h['TOT_EMP_22-24'], merged_l['TOT_EMP_22-24'], equal_var=False)
print(t_stat, p_welch)
t_stat, p_welch = stats.ttest_ind(merged_h['TOT_EMP_22-24'], merged_m['TOT_EMP_22-24'], equal_var=False)
print(t_stat, p_welch)
t_stat, p_welch = stats.ttest_ind(merged_l['TOT_EMP_22-24'], merged_m['TOT_EMP_22-24'], equal_var=False)
print(t_stat, p_welch)

for df, l in zip(mergeds, labels):
    df = df['TOT_EMP_21-24']
    print(f"In Year 21-24, AIOE-{l}'s mean is {df.mean():.4f}, median is {df.median():.4f}, standard deviation is {df.std():.4f}")
t_stat, p_welch = stats.ttest_ind(merged_h['TOT_EMP_21-24'], merged_l['TOT_EMP_21-24'], equal_var=False)
print(t_stat, p_welch)
t_stat, p_welch = stats.ttest_ind(merged_h['TOT_EMP_21-24'], merged_m['TOT_EMP_21-24'], equal_var=False)
print(t_stat, p_welch)
t_stat, p_welch = stats.ttest_ind(merged_l['TOT_EMP_21-24'], merged_m['TOT_EMP_21-24'], equal_var=False)
print(t_stat, p_welch)

In Year 22, AIOE-H's mean is 0.0407, median is 0.0383, standard deviation is 0.1150
In Year 22, AIOE-M's mean is 0.0503, median is 0.0337, standard deviation is 0.1376
In Year 22, AIOE-L's mean is 0.0315, median is 0.0285, standard deviation is 0.1425
-0.8199591069311627 0.41274746065322404
-1.4054840619707407 0.16084547156208828
0.648102881541774 0.5173894350118925

In Year 23, AIOE-H's mean is 0.0260, median is 0.0298, standard deviation is 0.0765
In Year 23, AIOE-M's mean is 0.0247, median is 0.0169, standard deviation is 0.1004
In Year 23, AIOE-L's mean is -0.0031, median is -0.0009, standard deviation is 0.1001
0.16659020363205038 0.8677736738609632
-2.9168433214725424 0.0037771286839314655
2.979364524296766 0.003117863593643061

In Year 24, AIOE-H's mean is 0.0124, median is 0.0175, standard deviation is 0.0903
In Year 24, AIOE-M's mean is 0.0058, median is 0.0122, standard deviation is 0.0952
In Year 24, AIOE-L's mean is -0.0023, median is 0.0057, standard deviation is 0.1208
0.

In [13]:
t_stat, p_welch = stats.ttest_ind(merged_h['dispersion_ratio_change_22-24'], merged_l['dispersion_ratio_change_22-24'], equal_var=False)
print(t_stat, p_welch)
t_stat, p_welch = stats.ttest_ind(merged_h['dispersion_ratio_change_22-24'], merged_m['dispersion_ratio_change_22-24'], equal_var=False)
print(t_stat, p_welch)
t_stat, p_welch = stats.ttest_ind(merged_l['dispersion_ratio_change_22-24'], merged_m['dispersion_ratio_change_22-24'], equal_var=False)
print(t_stat, p_welch)

2.2417856757244556 0.025733232907157823
0.4588597297689474 0.6465374690504463
-1.5215281433031496 0.12884650962624455


In [14]:
merged_df.columns

Index(['OCC_CODE', 'OCC_TITLE', 'AIOE', 'TOT_EMP_21', 'A_PCT25_21',
       'A_MEDIAN_21', 'A_PCT75_21', 'TOT_EMP_22', 'A_PCT25_22', 'A_MEDIAN_22',
       'A_PCT75_22', 'TOT_EMP_23', 'A_PCT25_23', 'A_MEDIAN_23', 'A_PCT75_23',
       'TOT_EMP_24', 'A_PCT25_24', 'A_MEDIAN_24', 'A_PCT75_24', 'AIOE_CAT',
       'TOT_EMP_YoY_22', 'TOT_EMP_YoY_23', 'TOT_EMP_YoY_24', 'TOT_EMP_22-24',
       'TOT_EMP_21-24', 'Product_A_PCT25_21', 'Product_A_PCT25_22',
       'A_PCT25_YoY_22', 'Product_A_PCT25_23', 'A_PCT25_YoY_23',
       'Product_A_PCT25_24', 'A_PCT25_YoY_24', 'A_PCT25_22-24',
       'A_PCT25_21-24', 'Product_A_MEDIAN_21', 'Product_A_MEDIAN_22',
       'A_MEDIAN_YoY_22', 'Product_A_MEDIAN_23', 'A_MEDIAN_YoY_23',
       'Product_A_MEDIAN_24', 'A_MEDIAN_YoY_24', 'A_MEDIAN_22-24',
       'A_MEDIAN_21-24', 'Product_A_PCT75_21', 'Product_A_PCT75_22',
       'A_PCT75_YoY_22', 'Product_A_PCT75_23', 'A_PCT75_YoY_23',
       'Product_A_PCT75_24', 'A_PCT75_YoY_24', 'A_PCT75_22-24',
       'A_PCT75_21-24

In [15]:
emp_growth_22_24_col = list(merged_df['TOT_EMP_22-24'])

print(len(aioe_col))

emp_growth_25_percentile = np.percentile(emp_growth_22_24_col, 25)
emp_growth_75_percentile = np.percentile(emp_growth_22_24_col, 75)

emp_growth_33_percentile = np.percentile(emp_growth_22_24_col, 33.3)
emp_growth_67_percentile = np.percentile(emp_growth_22_24_col, 66.7)

print(emp_growth_25_percentile, emp_growth_75_percentile)
print(emp_growth_33_percentile, emp_growth_67_percentile)

661
-0.05070046697798536 0.09296198720361315
-0.024370059008972562 0.06834922627251852


In [16]:
merged_df['EMP_GROWTH_CAT'] = 'M'

# merged_df.loc[merged_df['TOT_EMP_22-24'] <= emp_growth_25_percentile, 'EMP_GROWTH_CAT'] = 'L'
# merged_df.loc[merged_df['TOT_EMP_22-24'] > emp_growth_75_percentile, 'EMP_GROWTH_CAT'] = 'H'

merged_df.loc[merged_df['TOT_EMP_22-24'] <= emp_growth_33_percentile, 'EMP_GROWTH_CAT'] = 'L'
merged_df.loc[merged_df['TOT_EMP_22-24'] > emp_growth_67_percentile, 'EMP_GROWTH_CAT'] = 'H'

merged_df.head()

,OCC_CODE,OCC_TITLE,AIOE,TOT_EMP_21,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,TOT_EMP_22,A_PCT25_22,A_MEDIAN_22,...,Product_A_PCT75_24,A_PCT75_YoY_24,A_PCT75_22-24,A_PCT75_21-24,dispersion_ratio_21,dispersion_ratio_22,dispersion_ratio_23,dispersion_ratio_24,dispersion_ratio_change_22-24,EMP_GROWTH_CAT
0,11-1021,General and Operations Managers,0.574877,2984920,60690.0,97970.0,151750.0,3376680,62070.0,98100.0,...,5.883109e+11,0.023957,0.061918,0.081582,0.929468,0.942813,0.939080,0.941914,-0.000954,M
1,11-2011,Advertising and Promotions Managers,1.294387,22520,94020.0,127150.0,172210.0,22010,85020.0,127830.0,...,3.767827e+09,-0.052830,0.032674,0.036932,0.614943,0.687632,0.756199,0.729206,0.060460,L
2,11-2021,Marketing Managers,1.315032,278690,100010.0,135030.0,192520.0,328570,103060.0,140040.0,...,8.126158e+10,0.014808,0.063215,0.096406,0.685107,0.681734,0.634437,0.620195,-0.090268,H
3,11-2022,Sales Managers,1.266280,453800,84450.0,127490.0,173010.0,536390,87910.0,130600.0,...,1.216415e+11,0.024352,0.116969,0.164615,0.694643,0.708116,0.763244,0.764740,0.079964,H
4,11-3021,Computer and Information Systems Managers,1.059853,485190,122820.0,159010.0,198750.0,533220,127180.0,164070.0,...,1.396716e+11,0.010138,0.040269,0.087899,0.477517,0.491680,0.485399,0.478213,-0.027391,H


In [17]:
dispersion_change_22_24_col = list(merged_df['dispersion_ratio_change_22-24'])

print(len(dispersion_change_22_24_col))

dispersion_change_25_percentile = np.percentile(dispersion_change_22_24_col, 25)
dispersion_change_75_percentile = np.percentile(dispersion_change_22_24_col, 75)

dispersion_change_33_percentile = np.percentile(dispersion_change_22_24_col, 33.3)
dispersion_change_67_percentile = np.percentile(dispersion_change_22_24_col, 66.7)

print(dispersion_change_25_percentile, dispersion_change_75_percentile)
print(dispersion_change_33_percentile, dispersion_change_67_percentile)

661
-0.11891358540992114 0.03873273116852016
-0.0965951564649405 0.007069390017414247


In [18]:
merged_df['DISP_CONTRA_CAT'] = 'M'

# merged_df.loc[merged_df['dispersion_ratio_change_22-24'] <= dispersion_change_25_percentile, 'DISP_CONTRA_CAT'] = 'L'
# merged_df.loc[merged_df['dispersion_ratio_change_22-24'] > dispersion_change_75_percentile, 'DISP_CONTRA_CAT'] = 'H'

merged_df.loc[merged_df['dispersion_ratio_change_22-24'] <= dispersion_change_33_percentile, 'DISP_CONTRA_CAT'] = 'L'
merged_df.loc[merged_df['dispersion_ratio_change_22-24'] > dispersion_change_67_percentile, 'DISP_CONTRA_CAT'] = 'H'

merged_df.head()

,OCC_CODE,OCC_TITLE,AIOE,TOT_EMP_21,A_PCT25_21,A_MEDIAN_21,A_PCT75_21,TOT_EMP_22,A_PCT25_22,A_MEDIAN_22,...,A_PCT75_YoY_24,A_PCT75_22-24,A_PCT75_21-24,dispersion_ratio_21,dispersion_ratio_22,dispersion_ratio_23,dispersion_ratio_24,dispersion_ratio_change_22-24,EMP_GROWTH_CAT,DISP_CONTRA_CAT
0,11-1021,General and Operations Managers,0.574877,2984920,60690.0,97970.0,151750.0,3376680,62070.0,98100.0,...,0.023957,0.061918,0.081582,0.929468,0.942813,0.939080,0.941914,-0.000954,M,M
1,11-2011,Advertising and Promotions Managers,1.294387,22520,94020.0,127150.0,172210.0,22010,85020.0,127830.0,...,-0.052830,0.032674,0.036932,0.614943,0.687632,0.756199,0.729206,0.060460,L,H
2,11-2021,Marketing Managers,1.315032,278690,100010.0,135030.0,192520.0,328570,103060.0,140040.0,...,0.014808,0.063215,0.096406,0.685107,0.681734,0.634437,0.620195,-0.090268,H,M
3,11-2022,Sales Managers,1.266280,453800,84450.0,127490.0,173010.0,536390,87910.0,130600.0,...,0.024352,0.116969,0.164615,0.694643,0.708116,0.763244,0.764740,0.079964,H,H
4,11-3021,Computer and Information Systems Managers,1.059853,485190,122820.0,159010.0,198750.0,533220,127180.0,164070.0,...,0.010138,0.040269,0.087899,0.477517,0.491680,0.485399,0.478213,-0.027391,H,M


In [19]:
detailed_category_df = merged_df[['OCC_CODE', 'OCC_TITLE', 'AIOE', 'AIOE_CAT', 'TOT_EMP_24', 'TOT_EMP_22-24', 'EMP_GROWTH_CAT', 'A_MEDIAN_24', 'dispersion_ratio_24', 'dispersion_ratio_change_22-24', 'DISP_CONTRA_CAT']]

detailed_category_df

,OCC_CODE,OCC_TITLE,AIOE,AIOE_CAT,TOT_EMP_24,TOT_EMP_22-24,EMP_GROWTH_CAT,A_MEDIAN_24,dispersion_ratio_24,dispersion_ratio_change_22-24,DISP_CONTRA_CAT
0,11-1021,General and Operations Managers,0.574877,M,3584420,0.061522,M,102950.0,0.941914,-0.000954,M
1,11-2011,Advertising and Promotions Managers,1.294387,H,21100,-0.041345,L,126960.0,0.729206,0.060460,H
2,11-2021,Marketing Managers,1.315032,H,384980,0.171683,H,161030.0,0.620195,-0.090268,M
3,11-2022,Sales Managers,1.266280,H,603710,0.125506,H,138060.0,0.764740,0.079964,H
4,11-3021,Computer and Information Systems Managers,1.059853,H,645970,0.211451,H,171200.0,0.478213,-0.027391,M
...,...,...,...,...,...,...,...,...,...,...,...
656,53-7071,Gas Compressor and Gas Pumping Station Operators,-0.690999,M,5110,0.366310,H,71510.0,0.427493,-0.188183,L
657,53-7072,"Pump Operators, Except Wellhead Pumpers",-0.842293,M,12600,0.159154,H,60020.0,0.464678,-0.118909,L
658,53-7073,Wellhead Pumpers,-1.091197,L,17350,0.192440,H,70010.0,0.375232,-0.250720,L
659,53-7081,Refuse and Recyclable Material Collectors,-1.283373,L,139180,0.052480,M,48350.0,0.469080,0.033589,H


In [20]:
category_cnt_df = merged_df[['AIOE_CAT', 'EMP_GROWTH_CAT', 'DISP_CONTRA_CAT']].value_counts().reset_index()

category_emp_df = merged_df.groupby(['AIOE_CAT', 'EMP_GROWTH_CAT', 'DISP_CONTRA_CAT'])['TOT_EMP_24'].sum().reset_index()

category_df = category_cnt_df.merge(category_emp_df, how='inner', on=['AIOE_CAT', 'EMP_GROWTH_CAT', 'DISP_CONTRA_CAT'])

category_df.sort_values(['EMP_GROWTH_CAT', 'DISP_CONTRA_CAT'])

,AIOE_CAT,EMP_GROWTH_CAT,DISP_CONTRA_CAT,count,TOT_EMP_24
2,M,H,H,38,6681740
10,H,H,H,26,4069610
26,L,H,H,10,3080200
1,M,H,L,43,4662630
15,L,H,L,21,3193030
21,H,H,L,14,3264880
6,M,H,M,33,7054500
12,H,H,M,24,4432970
24,L,H,M,11,785570
4,M,L,H,36,2700110


In [21]:
category_df[category_df['AIOE_CAT'] == 'H']

,AIOE_CAT,EMP_GROWTH_CAT,DISP_CONTRA_CAT,count,TOT_EMP_24
10,H,H,H,26,4069610
11,H,M,M,26,7183610
12,H,H,M,24,4432970
17,H,M,H,20,2307040
18,H,L,H,18,3040410
21,H,H,L,14,3264880
22,H,L,M,13,2458010
23,H,L,L,13,787120
25,H,M,L,11,2199310


In [22]:
category_df[category_df['AIOE_CAT'] == 'L']

,AIOE_CAT,EMP_GROWTH_CAT,DISP_CONTRA_CAT,count,TOT_EMP_24
9,L,L,L,30,1999170
13,L,L,H,22,1600680
14,L,L,M,22,780930
15,L,H,L,21,3193030
16,L,M,L,20,9013490
19,L,M,M,15,5771140
20,L,M,H,15,4115830
24,L,H,M,11,785570
26,L,H,H,10,3080200


In [23]:
HHH_group = detailed_category_df[(detailed_category_df['AIOE_CAT'] == 'H') & (detailed_category_df['EMP_GROWTH_CAT'] == 'H') & (detailed_category_df['DISP_CONTRA_CAT'] == 'H')]

HHH_group.sort_values(['TOT_EMP_24', 'A_MEDIAN_24'], ascending=False)

,OCC_CODE,OCC_TITLE,AIOE,AIOE_CAT,TOT_EMP_24,TOT_EMP_22-24,EMP_GROWTH_CAT,A_MEDIAN_24,dispersion_ratio_24,dispersion_ratio_change_22-24,DISP_CONTRA_CAT
3,11-2022,Sales Managers,1.266280,H,603710,0.125506,H,138060.0,0.764740,0.079964,H
22,11-9111,Medical and Health Services Managers,1.260259,H,565840,0.186869,H,117960.0,0.626144,0.062631,H
42,13-1151,Training and Development Specialists,1.198409,H,436610,0.189090,H,65850.0,0.647684,0.091237,H
69,17-2051,Civil Engineers,1.282867,H,355410,0.155542,H,99590.0,0.497038,0.034566,H
75,17-2112,Industrial Engineers,1.137870,H,350230,0.089701,H,101140.0,0.450564,0.101822,H
13,11-9021,Construction Managers,1.029139,H,348330,0.148770,H,106980.0,0.522060,0.012396,H
30,13-1031,"Claims Adjusters, Examiners, and Investigators",1.274024,H,305020,0.069233,H,76790.0,0.467379,0.197119,H
25,11-9141,"Property, Real Estate, and Community Associati...",1.215396,H,296640,0.136029,H,66700.0,0.693103,0.083778,H
37,13-1081,Logisticians,1.297303,H,235640,0.160960,H,80880.0,0.511993,0.007864,H
10,11-3121,Human Resources Managers,1.320417,H,215520,0.188355,H,140030.0,0.602514,0.073852,H


In [24]:
detailed_category_df[(detailed_category_df['AIOE_CAT'] == 'H') & (detailed_category_df['EMP_GROWTH_CAT'] == 'M') & (detailed_category_df['DISP_CONTRA_CAT'] == 'H')]

,OCC_CODE,OCC_TITLE,AIOE,AIOE_CAT,TOT_EMP_24,TOT_EMP_22-24,EMP_GROWTH_CAT,A_MEDIAN_24,dispersion_ratio_24,dispersion_ratio_change_22-24,DISP_CONTRA_CAT
16,11-9033,"Education Administrators, Postsecondary",1.461149,H,176420,0.056028,M,103960.0,0.587341,0.067253,H
36,13-1075,Labor Relations Specialists,1.334834,H,64590,0.038424,M,93500.0,0.592727,0.080453,H
59,15-2031,Operations Research Analysts,1.400020,H,107760,0.025504,M,91290.0,0.626684,0.076109,H
61,17-1011,"Architects, Except Landscape and Naval",1.047142,H,111140,0.033957,M,96690.0,0.488055,0.011520,H
95,19-1011,Animal Scientists,1.041358,H,2470,-0.019841,M,79120.0,0.870071,0.303134,H
117,19-3022,Survey Researchers,1.426530,H,7720,-0.020305,M,63380.0,0.619438,0.026056,H
134,21-1013,Marriage and Family Therapists,1.367434,H,65870,0.061050,M,63780.0,0.571025,0.059459,H
150,23-2011,Paralegals and Legal Assistants,1.292055,H,367220,0.063666,M,61010.0,0.493198,0.016974,H
158,25-1042,"Biological Science Teachers, Postsecondary",1.227677,H,53250,0.066707,M,83460.0,0.734484,0.011138,H
159,25-1043,"Forestry and Conservation Science Teachers, Po...",1.260278,H,1310,0.031496,M,100830.0,0.433403,0.060699,H
